Groundwater | Transport Modeling Track

# Step 5: Calibration — From Observations to Parameters

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In NB4 you built a transient GWF+GWE model with default transport parameters ($\alpha_L = 10$ m, $n_e = 0.20$). The model produced seasonal temperature dynamics, but are the parameters correct? In this notebook you will calibrate them against synthetic temperature observations and an independent tracer test — the transport analogue of the flow calibration you did in flow NB5.

| **Core content** | ~75 minutes |
|:--|:--|
| **Optional: Exercises & advanced topics** | +30 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Assess** a temperature observation network and extract synthetic monitoring data
2. **Analyse** a conservative tracer breakthrough curve using temporal moment analysis *(exercise)*
3. **Compute** transport calibration metrics (temperature RMSE, amplitude ratio, phase lag)
4. **Perform** a manual dispersivity ($\alpha_L$) sweep guided by independent tracer test estimates
5. **Interpret** calibration improvement by comparing before/after temperature time series
6. **Explain** which parameters transfer from a heat calibration to a solute transport model

## Prerequisites

Before starting this notebook, you should have:
- **Completed [4_model_implementation.ipynb](4_model_implementation.ipynb)** — you need a working transient GWF+GWE model
- **Understood transport parameters** from NB2 (dispersivity, porosity, retardation)
- **Observed seasonal dynamics** in NB4 (amplitude attenuation, phase lag with distance)

In [ ]:
# Setup - import required libraries
import sys
import os
import shutil
import pickle
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*GeoSeries.*')

# Scientific computing
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# FloPy - MODFLOW 6 GWF
import flopy
from flopy.mf6 import MFSimulation, ModflowTdis, ModflowIms
from flopy.mf6 import ModflowGwf, ModflowGwfdisv
from flopy.mf6 import ModflowGwfnpf, ModflowGwfic, ModflowGwfoc
from flopy.mf6 import ModflowGwfrcha, ModflowGwfchd, ModflowGwfwel, ModflowGwfriv

# FloPy - MODFLOW 6 GWE
from flopy.mf6 import ModflowGwe, ModflowGwedisv
from flopy.mf6 import ModflowGweest, ModflowGwecnd, ModflowGweadv
from flopy.mf6 import ModflowGwessm, ModflowGweesl, ModflowGweic, ModflowGweoc
from flopy.mf6 import ModflowGwfgwe

# Course utilities
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import load_base_simulation
from plot_utils import quick_model_plot
import calibration_utils as cu
import tracer_test_utils as ttu
from generate_tracer_test_data import generate_tracer_test_data

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

print(f'FloPy version: {flopy.__version__}')


In [ ]:
# Define workspace paths
DATA_DIR = get_default_data_folder()
NB4_WS = os.path.join(DATA_DIR, 'transport_notebook4_model')
CALIB_WS = os.path.join(DATA_DIR, 'transport_calibration')

# ── Skip/rerun control ──────────────────────────────────────
# Set FORCE_RERUN = True to rebuild and re-run everything from scratch.
# When False, existing simulation results are reused (saves ~20 min).
FORCE_RERUN = False

cache_file = os.path.join(CALIB_WS, '_nb5_cache.pkl')
if FORCE_RERUN or not os.path.exists(cache_file):
    # Fresh start: copy NB4 workspace
    if os.path.exists(CALIB_WS):
        shutil.rmtree(CALIB_WS)
    shutil.copytree(NB4_WS, CALIB_WS)
    print(f'Calibration workspace: {CALIB_WS} (fresh copy from NB4)')
else:
    print(f'Calibration workspace: {CALIB_WS} (cached results found — skipping rebuild)')
    print(f'  Set FORCE_RERUN = True above to rebuild from scratch')

print(f'Data directory: {DATA_DIR}')
print(f'NB4 transport model: {NB4_WS}')

In [ ]:
# Load flow simulation from the copied workspace
sim = load_base_simulation(CALIB_WS)
gwf = sim.get_model('gwf_limmat')
gwe = sim.get_model('gwe_limmat')

# --- Extract grid ---
disv = gwf.get_package('disv')
nlay = disv.nlay.get_data()
ncpl = disv.ncpl.get_data()
gridprops = {
    'ncpl': ncpl,
    'nvert': disv.nvert.get_data(),
    'vertices': disv.vertices.get_data(),
    'cell2d': disv.cell2d.get_data(),
}
modelgrid = gwf.modelgrid

# --- Extract geometry ---
model_top = disv.top.get_data()
model_bottom = disv.botm.get_data()       # (nlay, ncpl)
idomain = disv.idomain.get_data()          # (nlay, ncpl)

# --- Extract hydraulic properties ---
npf = gwf.get_package('npf')
k_array = npf.k.get_data()                # (nlay, ncpl)
k33_array = npf.k33.get_data()            # (nlay, ncpl)

# --- Extract boundary condition data ---
riv_spd_raw = gwf.get_package('riv').stress_period_data.get_data(0)
chd_spd_raw = gwf.get_package('chd').stress_period_data.get_data(0)
wel_spd_raw = gwf.get_package('wel').stress_period_data.get_data(0)
recharge_array = gwf.get_package('rcha').recharge.get_data(0)

# --- Extract initial heads ---
hds_path = os.path.join(CALIB_WS, 'gwf_limmat.hds')
initial_heads = flopy.utils.HeadFile(hds_path).get_data()  # (nlay, ncpl)

# --- Cell centres ---
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters
active_mask = idomain[0].flatten() > 0  # layer 0 active cells

# --- Load boundary GIS ---
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

print(f'Model loaded from {CALIB_WS}:')
print(f'  Grid: {nlay} x {ncpl} = {nlay * ncpl} DISV cells, {nlay} layers')
print(f'  K range: {k_array.min():.1f} to {k_array.max():.1f} m/day')
print(f'  RIV entries: {len(riv_spd_raw)}, CHD: {len(chd_spd_raw)}, WEL: {len(wel_spd_raw)}')

In [ ]:
# Classify river cells as Limmat or Sihl (replicates NB4 pattern)
rivers_path = download_named_file(name='rivers', data_type='gis')
rivers_gdf = gpd.read_file(rivers_path)
rivers_gdf = rivers_gdf[rivers_gdf['GEWAESSERNAME'].isin(['Sihl', 'Limmat'])]

limmat_poly = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Limmat'].union_all()
sihl_poly = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()

is_limmat = []
n_limmat = n_sihl = 0
for entry in riv_spd_raw:
    cell_id = entry['cellid']
    cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
    pt = Point(xc[cell_idx], yc[cell_idx])
    if limmat_poly.contains(pt):
        is_limmat.append(True)
        n_limmat += 1
    elif sihl_poly.contains(pt):
        is_limmat.append(False)
        n_sihl += 1
    else:
        d_lim = pt.distance(limmat_poly)
        d_sih = pt.distance(sihl_poly)
        is_limmat.append(d_lim <= d_sih)
        if d_lim <= d_sih:
            n_limmat += 1
        else:
            n_sihl += 1

print(f'River classification: Limmat={n_limmat}, Sihl={n_sihl}, Total={len(is_limmat)}')

In [ ]:
# Load measured temperature forcing data (same as NB4)
import calendar

# River Limmat: monthly mean water temperature
limmat_csv = os.path.join(DATA_DIR, 'river_temperature_limmat_monthly.csv')
df_limmat = pd.read_csv(limmat_csv)

# River Sihl: monthly mean water temperature
sihl_csv = os.path.join(DATA_DIR, 'river_temperature_sihl_monthly.csv')
df_sihl = pd.read_csv(sihl_csv)

# Air temperature for recharge
air_csv = os.path.join(DATA_DIR, 'air_temperature_fluntern.csv')
df_air = pd.read_csv(air_csv, parse_dates=['date'])
df_air['year'] = df_air['date'].dt.year
df_air['month'] = df_air['date'].dt.month
air_monthly_all = df_air.groupby(['year', 'month'])['temperature'].mean().reset_index()

# Find common complete years
lim_complete = set(df_limmat.groupby('year').filter(lambda x: len(x) == 12)['year'])
sih_complete = set(df_sihl.groupby('year').filter(lambda x: len(x) == 12)['year'])
air_complete = set(air_monthly_all.groupby('year').filter(lambda x: len(x) == 12)['year'])
common_years = sorted(lim_complete & sih_complete & air_complete)
year_start, year_end = common_years[0], common_years[-1]

df_limmat = df_limmat[df_limmat['year'].isin(common_years)].sort_values(['year', 'month'])
df_sihl = df_sihl[df_sihl['year'].isin(common_years)].sort_values(['year', 'month'])
T_limmat_monthly = df_limmat['mean'].values
T_sihl_monthly = df_sihl['mean'].values

air_filtered = air_monthly_all[air_monthly_all['year'].isin(common_years)].sort_values(['year', 'month'])
T_recharge_monthly = np.maximum(air_filtered['temperature'].values, 0.5)

nper = len(T_limmat_monthly)
nyears = len(common_years)
month_labels = pd.date_range(f'{year_start}-01', periods=nper, freq='MS')

# Thermal parameters (same as NB4)
T_background = 10.5
T_lateral = 10.5
T_chd = 10.5

# Solid thermal properties
rho_s, c_s = 2650.0, 880.0
rho_w, c_w = 1000.0, 4184.0
lambda_w, lambda_s = 0.60, 2.0
ktw = lambda_w * 86400
kts = lambda_s * 86400

print(f'Temperature forcing: {nper} months ({nyears} common years, {year_start}-{year_end})')

# --- Calibration period: last 2 years only ---
calib_years = common_years[-2:]
calib_nper = len(calib_years) * 12  # 24 stress periods
calib_start_idx = common_years.index(calib_years[0]) * 12  # position-based, handles gaps

T_limmat_calib = T_limmat_monthly[calib_start_idx:]
T_sihl_calib = T_sihl_monthly[calib_start_idx:]
T_recharge_calib = T_recharge_monthly[calib_start_idx:]
month_labels_calib = pd.date_range(f'{calib_years[0]}-01', periods=calib_nper, freq='MS')

print(f'Calibration period: {calib_years[0]}-{calib_years[-1]} ({calib_nper} months)')


In [ ]:
# --- ESL inputs (Energy Source Loading) ---
from shapely.geometry import Polygon as ShapelyPolygon

q_geo_Wm2 = 0.070      # [W/m^2] Swiss Molasse Basin
q_geo_daily = q_geo_Wm2 * 86400  # [J/(d*m^2)]
q_urban_Wm2 = 1.5       # [W/m^2] urban zone avg
q_urban_daily = q_urban_Wm2 * 86400  # [J/(d*m^2)]
urban_x_min = 2_678_500  # Eastern part of domain

# Compute cell areas from DISV vertices
verts_data = gridprops['vertices']
cell2d_data = gridprops['cell2d']
vert_dict = {}
for v in verts_data:
    vert_dict[v[0]] = (v[1], v[2])

cell_areas = np.zeros(ncpl)
cell_xc_esl = np.zeros(ncpl)
for rec in cell2d_data:
    icell = rec[0]
    cell_xc_esl[icell] = rec[1]
    nv = rec[3]
    vert_ids = rec[4:4+nv]
    poly_coords = [vert_dict[vid] for vid in vert_ids]
    cell_areas[icell] = ShapelyPolygon(poly_coords).area

# Build combined ESL: geothermal (bottom) + urban (top)
esl_spd = []
bot_lay = nlay - 1
for icell in range(ncpl):
    if idomain[bot_lay, icell] > 0:
        esl_spd.append([(bot_lay, icell), q_geo_daily * cell_areas[icell]])
for icell in range(ncpl):
    if idomain[0, icell] > 0 and cell_xc_esl[icell] > urban_x_min:
        esl_spd.append([(0, icell), q_urban_daily * cell_areas[icell]])

print(f'ESL: {len(esl_spd)} entries (geothermal + urban)')

## 1 — Observation Network

### 1.1 Synthetic Temperature Observations

In flow NB5, we generated a "reference" K field and used it to create synthetic head observations. Here we do the same for temperature: we run a reference model with **known** transport parameters ($\alpha_L = 5$ m, $n_e = 0.18$) over the full record to establish equilibrated seasonal dynamics, then add measurement noise. The sweep models also run the full period — thermal equilibration takes many years, so a shorter spin-up would mask the dispersivity signal.

The reference parameters differ from NB4's defaults ($\alpha_L = 10$ m, $n_e = 0.20$), so the initial model will show a detectable mismatch — just like the initial K field mismatched the head observations in flow NB5.

In [ ]:
# ============================================================
# Generate reference temperature observations
# ============================================================
# Reference ("true") transport parameters — different from NB4 defaults
alpha_L_ref = 5.0     # True dispersivity [m] (NB4 default: 10)
n_e_ref = 0.18        # True porosity [-] (NB4 default: 0.20)

# --- Place 5 monitoring wells distributed across the aquifer ---
# Extract river cell indices for exclusion
riv_cells = np.array([
    entry['cellid'][1] if isinstance(entry['cellid'], tuple) else entry['cellid']
    for entry in riv_spd_raw
])

# Reuse flow NB5's spatial distribution strategy (heads used only for well placement,
# not temperature — the function avoids river/boundary cells and spaces wells evenly)
syn_gdf = cu.generate_synthetic_observations(
    model_grid=modelgrid,
    true_heads=initial_heads[0],     # used for spatial placement only
    idomain=idomain[0:1],            # layer 0 active mask
    n_points=5,
    seed=42,
    exclude_cells=riv_cells,
    river_gdf=rivers_gdf,
    boundary_polygon=boundary_gdf.geometry.iloc[0],
    avoid_boundaries_m=100,
    exclude_north_of_river=True,
    min_distance_m=200,
    river_buffer_m=50,
)

# Rename wells and compute distance from nearest river cell
obs_well_ids = [f'TW-{i+1}' for i in range(len(syn_gdf))]
obs_cells = syn_gdf['cell_id'].values.tolist()

# Distance from nearest river cell (more meaningful than from one point)
riv_xc = xc[riv_cells]
riv_yc = yc[riv_cells]
distance_m = []
for cell in obs_cells:
    d = np.sqrt((xc[cell] - riv_xc)**2 + (yc[cell] - riv_yc)**2)
    distance_m.append(d.min())

obs_gdf = gpd.GeoDataFrame({
    'obs_id': obs_well_ids,
    'x': syn_gdf['x'].values,
    'y': syn_gdf['y'].values,
    'distance_m': np.round(distance_m, 0),
    'cell_idx': obs_cells,
    'geometry': syn_gdf.geometry.values,
}, crs='EPSG:2056')

print(f'Monitoring wells placed:')
for _, row in obs_gdf.iterrows():
    print(f'  {row["obs_id"]}: ({row["x"]:.0f}, {row["y"]:.0f}), d={row["distance_m"]:.0f} m from nearest river cell')


In [ ]:
# ============================================================
# Build and run the REFERENCE model (alpha_L=5, n_e=0.18)
# ============================================================
REF_WS = os.path.join(CALIB_WS, 'reference_run')

# Time discretisation — full period for reference, calibration period for sweep
full_periods = []
for year in common_years:
    for month in range(1, 13):
        ndays = calendar.monthrange(year, month)[1]
        nstp = max(1, ndays // 5)
        full_periods.append((ndays, nstp, 1.0))


def build_and_run_transport(ws, alpha_L, n_e, n_periods, period_data,
                            t_limmat, t_sihl, t_recharge,
                            label='', strt_temperature=None):
    """Build GWF+GWE simulation and run."""
    _nper = n_periods
    _periods = period_data
    _t_lim = t_limmat
    _t_sih = t_sihl
    _t_rech = t_recharge

    sim_t = MFSimulation(sim_name='transport', sim_ws=ws)
    ModflowTdis(sim_t, nper=_nper, perioddata=_periods)

    # GWF model
    gwf_t = ModflowGwf(sim_t, modelname='gwf_t', save_flows=True, newtonoptions='NEWTON')
    ModflowGwfdisv(gwf_t, nlay=nlay, ncpl=ncpl, nvert=gridprops['nvert'],
                   vertices=gridprops['vertices'], cell2d=gridprops['cell2d'],
                   top=model_top, botm=model_bottom, idomain=idomain)
    ModflowGwfnpf(gwf_t, icelltype=[1] + [0] * (nlay - 1), k=k_array, k33=k33_array,
                  save_flows=True, save_specific_discharge=True)
    ModflowGwfic(gwf_t, strt=initial_heads)

    # Transient RIV with monthly temperatures
    riv_spd_transient = {}
    for per in range(_nper):
        T_lim = _t_lim[per]
        T_sih = _t_sih[per]
        riv_per = []
        for i, entry in enumerate(riv_spd_raw):
            T_riv = T_lim if is_limmat[i] else T_sih
            riv_per.append([entry['cellid'], entry['stage'], entry['cond'], entry['rbot'], T_riv])
        riv_spd_transient[per] = riv_per
    ModflowGwfriv(gwf_t, stress_period_data=riv_spd_transient,
                  auxiliary=['TEMPERATURE'], pname='RIV-1')

    # CHD
    chd_spd = [[entry['cellid'], entry['head'], T_chd] for entry in chd_spd_raw]
    ModflowGwfchd(gwf_t, stress_period_data={0: chd_spd},
                  auxiliary=['TEMPERATURE'], pname='CHD-1')

    # WEL
    wel_spd = [[entry['cellid'], entry['q'], T_lateral] for entry in wel_spd_raw]
    ModflowGwfwel(gwf_t, stress_period_data={0: wel_spd},
                  auxiliary=['TEMPERATURE'], pname='WEL-1')

    # RCHA
    rcha_aux = {}
    for per in range(_nper):
        temp_arr = np.where(recharge_array > 0, _t_rech[per], 0.0)
        rcha_aux[per] = [temp_arr]
    ModflowGwfrcha(gwf_t, recharge=recharge_array,
                   auxiliary='TEMPERATURE', aux=rcha_aux, pname='RCHA-1')

    ModflowGwfoc(gwf_t, head_filerecord='gwf_t.hds', budget_filerecord='gwf_t.cbc',
                 saverecord=[('HEAD', 'LAST'), ('BUDGET', 'ALL')])

    # GWE model
    gwe_t = ModflowGwe(sim_t, modelname='gwe_t')
    ModflowGwedisv(gwe_t, nlay=nlay, ncpl=ncpl, nvert=gridprops['nvert'],
                   vertices=gridprops['vertices'], cell2d=gridprops['cell2d'],
                   top=model_top, botm=model_bottom, idomain=idomain)
    ModflowGweest(gwe_t, porosity=n_e, density_solid=rho_s, heat_capacity_solid=c_s,
                  density_water=rho_w, heat_capacity_water=c_w, save_flows=True)
    ModflowGwecnd(gwe_t, alh=alpha_L, ath1=alpha_L / 10.0, atv=alpha_L / 10.0,
                  ktw=ktw, kts=kts)
    ModflowGweadv(gwe_t, scheme='TVD')
    ModflowGwessm(gwe_t, save_flows=True, sources=[
        ('RIV-1', 'AUX', 'TEMPERATURE'), ('CHD-1', 'AUX', 'TEMPERATURE'),
        ('WEL-1', 'AUX', 'TEMPERATURE'), ('RCHA-1', 'AUX', 'TEMPERATURE')])

    # ESL — Energy Source Loading (geothermal + urban heat)
    ModflowGweesl(gwe_t, stress_period_data={0: esl_spd}, save_flows=True)

    # GWE initial condition: warm-start or uniform background
    strt = strt_temperature if strt_temperature is not None else T_background
    ModflowGweic(gwe_t, strt=strt)
    ModflowGweoc(gwe_t, temperature_filerecord='gwe_t.ucn', budget_filerecord='gwe_t.cbc',
                 saverecord=[('TEMPERATURE', 'LAST'), ('BUDGET', 'ALL')])

    # Solvers
    ims_gwf = ModflowIms(sim_t, complexity='COMPLEX', outer_maximum=500, inner_maximum=300,
                         outer_dvclose=1e-2, inner_dvclose=1e-4,
                         linear_acceleration='BICGSTAB', under_relaxation='DBD',
                         under_relaxation_theta=0.7, under_relaxation_kappa=0.1,
                         under_relaxation_gamma=0.0, filename='gwf_t.ims')
    sim_t.register_ims_package(ims_gwf, [gwf_t.name])
    ims_gwe = ModflowIms(sim_t, complexity='MODERATE', outer_maximum=200, inner_maximum=100,
                         outer_dvclose=1e-4, inner_dvclose=1e-7,
                         linear_acceleration='BICGSTAB', filename='gwe_t.ims')
    sim_t.register_ims_package(ims_gwe, [gwe_t.name])
    ModflowGwfgwe(sim_t, exgtype='GWF6-GWE6', exgmnamea=gwf_t.name, exgmnameb=gwe_t.name)

    # Write and run
    sim_t.write_simulation()
    if label:
        print(f'Running {label}...')
    success, _ = sim_t.run_simulation(silent=True)
    return success, sim_t


def extract_temperature_timeseries(ws, obs_cells, last_n=None):
    """Extract monthly temperature time series at observation cell indices.

    Parameters
    ----------
    last_n : int, optional
        If given, extract only the last N time steps.
    """
    ucn_path = os.path.join(ws, 'gwe_t.ucn')
    temp_file = flopy.utils.HeadFile(ucn_path, text='TEMPERATURE')
    kstpkper = temp_file.get_kstpkper()
    if last_n is not None:
        kstpkper = kstpkper[-last_n:]
    ts = np.zeros((len(kstpkper), len(obs_cells)))
    for ip, ksp in enumerate(kstpkper):
        data = temp_file.get_data(kstpkper=ksp)[0].flatten()
        for ic, cell in enumerate(obs_cells):
            ts[ip, ic] = data[cell]
    return ts


In [ ]:
# ── Cache check: skip reference model if cached ──
cache_file = os.path.join(CALIB_WS, '_nb5_cache.pkl')

if not FORCE_RERUN and os.path.exists(cache_file):
    print('Loading cached reference model results...')
    with open(cache_file, 'rb') as f:
        _cache = pickle.load(f)
    ref_ts = _cache['ref_ts']
    obs_ts = _cache['obs_ts']
    success_ref = True
    print(f'  Loaded: ref_ts {ref_ts.shape}, obs_ts {obs_ts.shape}')
    print(f'  Set FORCE_RERUN = True to rebuild from scratch')
else:
    if os.path.exists(REF_WS):
        shutil.rmtree(REF_WS)
    os.makedirs(REF_WS)

    # Run reference model (full period, T_background IC → equilibrated observations)
    print(f'Building reference model (alpha_L=5, n_e=0.18, {nper} months, full period)...')
    print('This will take several minutes...')
    success_ref, sim_ref = build_and_run_transport(
        REF_WS, alpha_L_ref, n_e_ref,
        nper, full_periods,
        T_limmat_monthly, T_sihl_monthly, T_recharge_monthly,
        label='Reference model')

    if success_ref:
        # Extract only the last calib_nper time steps (equilibrated seasonal behavior)
        ref_ts = extract_temperature_timeseries(
            REF_WS, obs_gdf['cell_idx'].values, last_n=calib_nper)

        # Add noise (sigma = 0.05 degC, precision temperature logger)
        rng = np.random.default_rng(42)
        obs_ts = ref_ts + rng.normal(0, 0.05, ref_ts.shape)
        # Save cache for next run
        with open(cache_file, 'wb') as _f:
            pickle.dump({'ref_ts': ref_ts, 'obs_ts': obs_ts}, _f)
        print(f'Results cached to {cache_file}')

    else:
        print('Reference model FAILED — check mfsim.lst')
        ref_ts = None
        obs_ts = None


In [ ]:
# Display observation summary
obs_summary = obs_gdf[['obs_id', 'distance_m']].copy()
obs_summary['mean_T_obs'] = obs_ts.mean(axis=0).round(2)
obs_summary['amplitude_obs'] = (obs_ts.max(axis=0) - obs_ts.min(axis=0)).round(2)
display(obs_summary)

In [ ]:
# Checkpoint 1 — How many monitoring wells?
check_task_with_solution('task_t05_checkpoint_1')

In [ ]:
# Map of monitoring network
fig, ax = plt.subplots(figsize=(12, 10))
boundary_gdf.plot(ax=ax, facecolor='#f0f0f0', edgecolor='black', linewidth=1.5)
rivers_gdf.plot(ax=ax, facecolor='#a6cee3', edgecolor='#1f78b4', linewidth=0.5, alpha=0.7)

# Plot monitoring wells
ax.scatter(obs_gdf['x'], obs_gdf['y'], c='red', s=120, edgecolor='black',
           linewidth=1.5, zorder=5, label='Temperature monitoring wells')

for _, row in obs_gdf.iterrows():
    ax.annotate(f'{row["obs_id"]} ({row["distance_m"]:.0f}m)',
                (row['x'], row['y']), xytext=(8, 8), textcoords='offset points',
                fontsize=9, fontweight='bold')

ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_title('Temperature Monitoring Network')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Plot observed temperature time series
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(obs_well_ids)))

for i, (wid, color) in enumerate(zip(obs_well_ids, colors)):
    d = obs_gdf.iloc[i]['distance_m']
    ax.plot(month_labels_calib, obs_ts[:, i], '-', color=color, linewidth=1.2,
            label=f'{wid} (d={d:.0f} m)')

# River forcing (dashed)
ax.plot(month_labels_calib, T_limmat_calib, '--', color='gray', alpha=0.5,
        linewidth=1, label='Limmat river T')
ax.axhline(T_background, color='gray', linestyle=':', alpha=0.4)

ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Observed Temperature Time Series at Monitoring Wells')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Key observation: seasonal amplitude attenuates with distance from the river.')
print('Near-river wells track the river closely; distant wells are nearly flat at background.')

---

## 2 — Initial Assessment

The NB4 model used default parameters ($\alpha_L = 10$ m, $n_e = 0.20$). Let's see how well it reproduces the observed temperature time series.

> **Prediction:** Before running the comparison, consider: if the default $\alpha_L$ is too high, would you expect the simulated seasonal amplitude to be *larger* or *smaller* than observed?


In [ ]:
# Load NB4 default model output — extract calibration period only
ucn_path_default = os.path.join(CALIB_WS, 'gwe_limmat.ucn')
temp_file_default = flopy.utils.HeadFile(ucn_path_default, text='TEMPERATURE')
kstpkper_default = temp_file_default.get_kstpkper()

# Extract only the last calib_nper time steps (matching calibration period)
kstpkper_calib = kstpkper_default[-calib_nper:]
default_ts = np.zeros((len(kstpkper_calib), len(obs_cells)))
for ip, ksp in enumerate(kstpkper_calib):
    data = temp_file_default.get_data(kstpkper=ksp)[0].flatten()
    for ic, cell in enumerate(obs_cells):
        default_ts[ip, ic] = data[cell]

print(f'Loaded NB4 default output: last {default_ts.shape[0]} time steps (calibration period) at {default_ts.shape[1]} wells')

In [ ]:
# Compute initial temperature metrics (per-well and overall)
print('Initial assessment (NB4 defaults: alpha_L=10, n_e=0.20):')
print(f'{"Well":<8} {"RMSE (°C)":>10} {"ME (°C)":>10}')
print('-' * 30)

all_obs = []
all_sim = []
for i, wid in enumerate(obs_well_ids):
    obs_flat = obs_ts[:, i]
    sim_flat = default_ts[:, i]
    rmse = np.sqrt(np.mean((obs_flat - sim_flat)**2))
    me = np.mean(sim_flat - obs_flat)
    print(f'{wid:<8} {rmse:>10.3f} {me:>10.3f}')
    all_obs.extend(obs_flat)
    all_sim.extend(sim_flat)

all_obs = np.array(all_obs)
all_sim = np.array(all_sim)
initial_rmse = np.sqrt(np.mean((all_obs - all_sim)**2))
initial_me = np.mean(all_sim - all_obs)
print('-' * 30)
print(f'{"Overall":<8} {initial_rmse:>10.3f} {initial_me:>10.3f}')

In [ ]:
# Checkpoint 2 — Initial temperature RMSE
check_task_with_solution('task_t05_checkpoint_2')

In [ ]:
# Plot observed vs simulated temperature time series
fig, axes = plt.subplots(len(obs_well_ids), 1, figsize=(14, 3 * len(obs_well_ids)), sharex=True)

for i, (wid, ax) in enumerate(zip(obs_well_ids, axes)):
    d = obs_gdf.iloc[i]['distance_m']
    ax.plot(month_labels_calib, obs_ts[:, i], 'o', markersize=2, color='black', alpha=0.5, label='Observed')
    ax.plot(month_labels_calib, default_ts[:, i], '-', color='tab:red', linewidth=1.2,
            label=f'Simulated (aL=10, ne=0.20)')
    ax.set_ylabel('T (°C)')
    ax.set_title(f'{wid} (d={d:.0f} m)', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.suptitle('Initial Assessment — Default vs Observed Temperature', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Amplitude ratio and phase analysis per well
print('Seasonal amplitude comparison:')
print(f'{"Well":<8} {"d (m)":>6} {"A_obs (C)":>10} {"A_sim (C)":>10} {"Ratio":>8}')
print('-' * 46)

for i, wid in enumerate(obs_well_ids):
    d = obs_gdf.iloc[i]['distance_m']
    a_obs = obs_ts[:, i].max() - obs_ts[:, i].min()
    a_sim = default_ts[:, i].max() - default_ts[:, i].min()
    ratio = a_sim / a_obs if a_obs > 0.01 else float('nan')
    print(f'{wid:<8} {d:>6.0f} {a_obs:>10.3f} {a_sim:>10.3f} {ratio:>8.2f}')

print()
print('Ratio < 1 means simulated signal is OVER-DAMPED (alpha_L too high)')
print('Ratio > 1 means simulated signal is UNDER-DAMPED (alpha_L too low)')

### How to Read Temperature Calibration Metrics

| Pattern | Meaning | Likely parameter issue |
|---------|---------|----------------------|
| Sim amplitude too low (ratio < 1) | Over-damped signal | $\alpha_L$ too high |
| Sim amplitude too high (ratio > 1) | Under-damped signal | $\alpha_L$ too low |
| Sim phase lag too large | Signal too slow | $n_e$ too high (lower seepage velocity $v = q/n_e$) |
| Sim phase lag too small | Signal too fast | $n_e$ too low |
| Uniform bias across all wells | Wrong background temperature | Check boundary conditions |

> The default model over-damps the seasonal signal (amplitude ratio < 1), which motivates calibrating $\alpha_L$ downward. The tracer test (Section 3) provides an independent estimate.

---

## 3 — Tracer Test: Independent Parameter Estimate

In flow NB5, a **pumping test** provided an independent K estimate before PEST++ calibration. Here, a **conservative tracer test** provides independent estimates of $\alpha_L$ and $n_e$ before temperature calibration. The approach is directly parallel:

| Flow NB5 | Transport NB5 |
|----------|---------------|
| Pumping test data | Tracer test BTC data |
| Cooper-Jacob analysis | Temporal moment analysis |
| Estimate of $K$, $T$, $S$ | Estimate of $v$, $n_e$, $\alpha_L$ |
| Constrains $K$ before head calibration | Constrains $n_e$ before temperature calibration |

### 3.1 Moment Analysis Theory

#### 1D ADE pulse solution and moment formulas

For a pulse injection of mass $M$ into a cross-section $A$ at $x=0$, the **flux-averaged** breakthrough curve at distance $x$ is:

$$C_f(x,t) = \frac{M}{A} \cdot \frac{x}{n_e \sqrt{4\pi D_L t^3}} \cdot \exp\left(-\frac{(x - vt)^2}{4 D_L t}\right)$$

The temporal moments of this curve give transport parameters directly:

**Zeroth moment** (total mass passing the observation point):

$$M_0 = \int_0^\infty C_f(x, t)\, dt$$

**First moment** (mean arrival time):

$$\bar{t} = \frac{M_1}{M_0} = \frac{\int t \cdot C_f(x, t)\, dt}{M_0} \quad \Rightarrow \quad v = \frac{x}{\bar{t}}, \quad n_e = \frac{q}{v}$$

**Second moment** (temporal variance):

$$\sigma_t^2 = \frac{M_2}{M_0} - \bar{t}^2 \quad \Rightarrow \quad \sigma_x^2 = v^2 \cdot \sigma_t^2 \quad \Rightarrow \quad \alpha_L = \frac{\sigma_x^2}{2x}$$


### 3.2 Load Tracer Test Data

A conservative dye (fluorescein) tracer test was conducted in the Limmat Valley aquifer. The injection point is upstream, and three monitoring wells (MW-1, MW-2, MW-3) recorded breakthrough curves at 50, 150, and 300 m distance.

In [ ]:
# Generate synthetic tracer test data
tt_output_dir = os.path.join(DATA_DIR, 'tracer_test')
os.makedirs(tt_output_dir, exist_ok=True)
generate_tracer_test_data(tt_output_dir)

# Load the data
import json
tt_df = pd.read_csv(os.path.join(tt_output_dir, 'tracer_test_data.csv'))
with open(os.path.join(tt_output_dir, 'tracer_test_metadata.json')) as f:
    tt_meta = json.load(f)

q_darcy = tt_meta['darcy_flux_m_per_day']
M_inject = tt_meta['injected_mass_g']
A_cross = tt_meta['cross_sectional_area_m2']

print(f'Tracer: {tt_meta["tracer"]}')
print(f'Injected mass: {M_inject} g')
print(f'Cross-sectional area: {A_cross} m2')
print(f'Darcy flux: {q_darcy} m/d')
print(f'Wells: {list(tt_meta["observation_wells"].keys())}')

In [ ]:
# Plot raw breakthrough curves
fig, ax = ttu.plot_all_wells_raw(tt_df, title='Tracer Test — Breakthrough Curves')
plt.show()

### 3.3 Step-by-Step Moment Analysis

Let's work through the moment analysis for MW-1 ($x = 50$ m) step by step.

**Available variables:** `t_mw1` (time array), `c_mw1` (concentration array), `x_mw1` (distance), `q_darcy` (Darcy flux)

**Key function:** `ttu.temporal_moment(time, concentration, order=n)` computes the $n$-th temporal moment $M_n = \int t^n \, C(t) \, dt$ using trapezoidal integration.

In [ ]:
# ✏️ EXERCISE — Step 1: Compute moments and velocity for MW-1 (x = 50 m)
mw1 = tt_df[tt_df['well_id'] == 'MW-1']
t_mw1 = mw1['time_d'].values
c_mw1 = mw1['concentration_mg_L'].values
x_mw1 = mw1['distance_m'].iloc[0]

# ✏️ Zeroth moment: M0 = integral of C dt
M0 = ttu.temporal_moment(
    ...,  # ← time array (hint: t_mw1)
    ...,  # ← concentration array (hint: c_mw1)
    order=0,
)
print(f'M0 (zeroth moment) = {M0:.4f} mg*d/L')

# ✏️ First moment: M1 = integral of t*C dt
M1 = ttu.temporal_moment(..., ..., order=1)  # ← same arrays
print(f'M1 (first moment) = {M1:.4f} mg*d^2/L')

# ✏️ Mean arrival time and velocity
t_bar = ...  # ← M1 / M0
v_mw1 = ...  # ← x_mw1 / t_bar
print(f't_bar = M1/M0 = {t_bar:.2f} days')
print(f'v = x / t_bar = {x_mw1:.0f} / {t_bar:.2f} = {v_mw1:.2f} m/d')

In [ ]:
# Checkpoint — Verify velocity at MW-1
check_task_with_solution('task_t05_tt_checkpoint_1')

**Step 2 — Effective porosity** from Darcy flux and seepage velocity:

$$n_e = \frac{q}{v}$$

In [ ]:
# ✏️ EXERCISE — Step 2: Derive effective porosity from velocity + Darcy flux
# n_e = q / v
n_e_mw1 = ...  # ✏️ q_darcy / v_mw1
print(f'n_e = q / v = {q_darcy:.3f} / {v_mw1:.2f} = {n_e_mw1:.4f}')
print(f'  (true value: 0.18)')

**Step 3 — Dispersivity** from the second temporal moment:

$$\sigma_t^2 = \frac{M_2}{M_0} - \bar{t}^2, \quad \sigma_x^2 = v^2 \, \sigma_t^2, \quad \alpha_L = \frac{\sigma_x^2}{2x}$$

In [ ]:
# ✏️ EXERCISE — Step 3: Second moment → variance → dispersivity
M2 = ttu.temporal_moment(t_mw1, c_mw1, order=2)  # provided
print(f'M2 (second moment) = {M2:.4f} mg*d^3/L')

# ✏️ Temporal variance
sigma2_t = ...  # ← M2 / M0 - t_bar**2
print(f'sigma2_t = M2/M0 - t_bar^2 = {sigma2_t:.4f} d^2')

# ✏️ Spatial variance
sigma2_x = ...  # ← v_mw1**2 * sigma2_t
print(f'sigma2_x = v^2 * sigma2_t = {sigma2_x:.2f} m^2')

# ✏️ Longitudinal dispersivity
alpha_L_mw1 = ...  # ← sigma2_x / (2 * x_mw1)
print(f'alpha_L = sigma2_x / (2*x) = {sigma2_x:.2f} / {2*x_mw1:.0f} = {alpha_L_mw1:.2f} m')

In [ ]:
# Checkpoint — Verify dispersivity at MW-1
check_task_with_solution('task_t05_tt_checkpoint_2')

**Step 4 — Extend to all wells:**

`ttu.analyze_all_wells(df, q_darcy)` runs the full moment analysis on every well and returns a summary DataFrame. `ttu.summarize_results(results_df)` prints a formatted table.

In [ ]:
# ✏️ EXERCISE — Step 4: Extend moment analysis to all wells
tt_results, tt_params = ttu.analyze_all_wells(
    ...,  # ← tt_df
    ...,  # ← q_darcy
)
ttu.summarize_results(...)  # ← tt_results

# Plot fitted curves
fig, axes = ttu.plot_all_wells_fitted(tt_df, tt_params, tt_results, M_inject, A_cross)
plt.show()

# Store mean estimates for use in calibration
alpha_L_tracer = tt_results['alpha_L_m'].mean()
n_e_tracer = tt_results['n_e'].mean()
print(f'Tracer test estimates:')
print(f'  alpha_L = {alpha_L_tracer:.1f} m (mean of {len(tt_results)} wells)')
print(f'  n_e = {n_e_tracer:.3f} (mean of {len(tt_results)} wells)')

In [ ]:
# Checkpoint — Why does alpha_L vary across wells?
create_multiple_choice('task_t05_tt_checkpoint_3')

In [ ]:
# Fallback: use defaults if exercise was skipped
if 'alpha_L_tracer' not in dir():
    alpha_L_tracer = 5.0
    n_e_tracer = 0.18
    print("(Using default tracer estimates — complete Section 3 to derive these yourself)")
if 'tt_results' not in dir():
    tt_results = pd.DataFrame({
        'well_id': ['MW-1', 'MW-2', 'MW-3'],
        'v_md': [12.3, 12.1, 11.9],
        'n_e': [0.180, 0.183, 0.186],
        'alpha_L_m': [4.6, 5.1, 5.3],
    })

---

## 4 — Manual Calibration: Dispersivity Sweep

Armed with the tracer test estimates, we now calibrate the temperature model. The strategy:

| Parameter | Value | Source |
|-----------|-------|--------|
| $n_e$ | **Fixed** at tracer test estimate (~0.18) | Tracer test moment analysis |
| $\alpha_L$ | **Sweep**: 3, 5, 10 m | Tracer test suggests ~5 m |
| Thermal properties | Fixed | Literature values from NB2 |

This reduces calibration to a 1D search over $\alpha_L$ — the same simplification that flow NB5 achieved by calibrating K while keeping recharge fixed.

> In flow NB5, we used PEST++ for automated calibration. Here the transient runtime (~3.5 min per run) makes a Jacobian impractical. Manual sweeps with full-period runs are the practical choice — and they build intuition about parameter sensitivity.

In [ ]:
# ============================================================
# Calibration sweep: alpha_L = [3, 5, 10] m
# n_e fixed from tracer test
# ============================================================
alpha_L_values = [3, 5, 10]
n_e_calib = round(n_e_tracer, 2)  # Use rounded tracer test estimate

print(f'Calibration sweep:')
print(f'  alpha_L values: {alpha_L_values} m')
print(f'  n_e (fixed): {n_e_calib}')
print(f'  Number of runs: {len(alpha_L_values)}')
print(f'*** This will take ~7 minutes (reusing reference for alpha_L={alpha_L_ref}). ***')

In [ ]:
# Run calibration sweep (full-period models, reuse reference for alpha_L_ref)
sweep_results = []

for alpha_L_test in alpha_L_values:
    ws = os.path.join(CALIB_WS, f'sweep_aL{alpha_L_test}')

    if alpha_L_test == alpha_L_ref and n_e_calib == n_e_ref:
        # Reuse reference model output (same alpha_L and n_e)
        print(f'Running alpha_L={alpha_L_test} m, n_e={n_e_calib} ... (reusing reference)')
        sim_ts = ref_ts.copy()
    else:
        if os.path.exists(ws):
            shutil.rmtree(ws)
        os.makedirs(ws)

        success_t, sim_t = build_and_run_transport(
            ws, alpha_L_test, n_e_calib,
            nper, full_periods,
            T_limmat_monthly, T_sihl_monthly, T_recharge_monthly,
            label=f'alpha_L={alpha_L_test} m, n_e={n_e_calib}')

        if not success_t:
            print(f'  -> FAILED')
            sweep_results.append({
                'alpha_L': alpha_L_test,
                'RMSE': float('nan'),
                'ME': float('nan'),
                'Amp_ratio': float('nan'),
                'sim_ts': None,
            })
            continue

        # Extract last calib_nper months (equilibrated seasonal behaviour)
        sim_ts = extract_temperature_timeseries(
            ws, obs_gdf['cell_idx'].values, last_n=calib_nper)

    # Compute metrics
    obs_flat = obs_ts.flatten()
    sim_flat = sim_ts.flatten()
    rmse = np.sqrt(np.mean((obs_flat - sim_flat)**2))
    me = np.mean(sim_flat - obs_flat)

    # Amplitude ratio (mean across wells)
    amp_ratios = []
    for i in range(len(obs_well_ids)):
        a_obs = obs_ts[:, i].max() - obs_ts[:, i].min()
        a_sim = sim_ts[:, i].max() - sim_ts[:, i].min()
        if a_obs > 0.01:
            amp_ratios.append(a_sim / a_obs)
    mean_amp_ratio = np.mean(amp_ratios)

    sweep_results.append({
        'alpha_L': alpha_L_test,
        'RMSE': round(rmse, 4),
        'ME': round(me, 4),
        'Amp_ratio': round(mean_amp_ratio, 3),
        'sim_ts': sim_ts,
    })
    print(f'  -> RMSE = {rmse:.4f} C, Amp ratio = {mean_amp_ratio:.3f}')

print('Sweep complete!')


In [ ]:
# Summary table
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'sim_ts'} for r in sweep_results])
print('Calibration Sweep Results:')
print(results_df.to_string(index=False))

# Highlight best
best_idx = results_df['RMSE'].idxmin()
best_alpha = results_df.loc[best_idx, 'alpha_L']
best_rmse = results_df.loc[best_idx, 'RMSE']
print(f'Best: alpha_L = {best_alpha} m (RMSE = {best_rmse:.4f} C)')

In [ ]:
# Checkpoint — Which alpha_L gave the best RMSE?
create_multiple_choice('task_t05_checkpoint_best_alpha')

In [ ]:
# Plot observed vs best calibrated model
best_result = sweep_results[best_idx]
best_ts = best_result['sim_ts']

fig, axes = plt.subplots(len(obs_well_ids), 1, figsize=(14, 3 * len(obs_well_ids)), sharex=True)

for i, (wid, ax) in enumerate(zip(obs_well_ids, axes)):
    d = obs_gdf.iloc[i]['distance_m']
    ax.plot(month_labels_calib, obs_ts[:, i], 'o', markersize=2, color='black', alpha=0.5, label='Observed')
    ax.plot(month_labels_calib, default_ts[:, i], '-', color='tab:red', linewidth=0.8, alpha=0.4,
            label=f'Default (aL=10)')
    ax.plot(month_labels_calib, best_ts[:, i], '-', color='tab:blue', linewidth=1.2,
            label=f'Calibrated (aL={best_alpha:.0f})')
    ax.set_ylabel('T (°C)')
    ax.set_title(f'{wid} (d={d:.0f} m)', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.suptitle('Temperature Calibration — Before vs After', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Before/after metrics comparison
calibrated_rmse = best_rmse
print('Calibration improvement:')
print(f'  Initial RMSE:    {initial_rmse:.4f} C  (alpha_L=10, n_e=0.20)')
print(f'  Calibrated RMSE: {calibrated_rmse:.4f} C  (alpha_L={best_alpha:.0f}, n_e={n_e_calib})')
print(f'  Improvement:     {(1 - calibrated_rmse/initial_rmse)*100:.1f}%')

In [ ]:
# Checkpoint — Calibrated RMSE
check_task_with_solution('task_t05_checkpoint_3')

In [ ]:
# Amplitude ratio comparison — bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(obs_well_ids))
width = 0.3

# Compute amplitude ratios for default and calibrated
amp_default = []
amp_calib = []
for i in range(len(obs_well_ids)):
    a_obs = obs_ts[:, i].max() - obs_ts[:, i].min()
    a_def = default_ts[:, i].max() - default_ts[:, i].min()
    a_cal = best_ts[:, i].max() - best_ts[:, i].min()
    amp_default.append(a_def / a_obs if a_obs > 0.01 else 0)
    amp_calib.append(a_cal / a_obs if a_obs > 0.01 else 0)

ax.bar(x_pos - width/2, amp_default, width, label=f'Default (aL=10)', color='tab:red', alpha=0.7)
ax.bar(x_pos + width/2, amp_calib, width, label=f'Calibrated (aL={best_alpha:.0f})', color='tab:blue', alpha=0.7)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Perfect (ratio=1)')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{wid}\n({obs_gdf.iloc[i]["distance_m"]:.0f}m)' for i, wid in enumerate(obs_well_ids)])
ax.set_ylabel('Amplitude Ratio (sim/obs)')
ax.set_title('Seasonal Amplitude: Default vs Calibrated')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Discussion

The tracer test estimate ($\alpha_L \approx 5$ m) is confirmed by the temperature calibration — the RMSE minimum falls at $\alpha_L = 5$ m. **Two independent lines of evidence converging on the same answer** builds confidence in the result.

This is the transport analogue of flow NB5, where the pumping test K estimate was confirmed by the automated PEST++ calibration. The key difference: here we used manual sweeps instead of PEST++, because each transient run takes ~2 minutes (making Jacobian computation impractical). In production work, you would use adjoint-state methods or surrogate models to automate this.

> **Note:** The forcing data contains a real warming trend (~0.4 °C/decade, see NB2). This does not affect the calibration here, but you will explore its sensitivity implications in NB7.


---

## 5 — Assessment

In [ ]:
# Energy balance verification on the best calibrated model
# If best_alpha matches reference, use reference workspace
best_ws = os.path.join(CALIB_WS, f'sweep_aL{int(best_alpha)}')
if not os.path.exists(best_ws):
    best_ws = REF_WS  # alpha_L=5 reused reference output
sim_best = flopy.mf6.MFSimulation.load(sim_ws=best_ws, verbosity_level=0)
gwe_best = sim_best.get_model('gwe_t')

try:
    budget = gwe_best.output.budget()
    record_names = budget.get_unique_record_names()
    last_kstpkper = budget.get_kstpkper()[-1]

    print('GWE Energy Budget (last time step):')
    print(f'{"Component":<25s} {"Inflow":>15s} {"Outflow":>15s} {"Net":>15s}')
    print('-' * 70)

    total_in = 0.0
    total_out = 0.0

    for name in record_names:
        name_str = name.decode().strip() if isinstance(name, bytes) else str(name).strip()
        data = budget.get_data(text=name, kstpkper=last_kstpkper)
        if len(data) > 0:
            d = data[0]
            vals = d['q'] if hasattr(d, 'dtype') and d.dtype.names and 'q' in d.dtype.names else d.flatten()
            inflow = float(vals[vals > 0].sum())
            outflow = float(vals[vals < 0].sum())
            total_in += inflow
            total_out += abs(outflow)
            if abs(inflow) > 0 or abs(outflow) > 0:
                print(f'{name_str:<25s} {inflow:>15.1f} {outflow:>15.1f} {inflow + outflow:>15.1f}')

    print('-' * 70)
    print(f'{"TOTAL":<25s} {total_in:>15.1f} {-total_out:>15.1f} {total_in - total_out:>15.1f}')
    if (total_in + total_out) > 0:
        pct_error = 200 * abs(total_in - total_out) / (total_in + total_out)
        print(f'{"PERCENT ERROR":<25s} {pct_error:>15.4f}%')

except Exception as e:
    print(f'Could not read GWE budget: {e}')

In [ ]:
# Checkpoint — Energy balance error
check_task_with_solution('task_t05_checkpoint_4')

<details>
<summary><strong>What transfers from heat to solute?</strong> (click to expand)</summary>

When you have a calibrated heat transport model and want to build a solute transport model, some parameters transfer directly while others must be replaced:

| Parameter | Transfers? | Notes |
|-----------|------------|-------|
| $\alpha_L$, $\alpha_T$ (dispersivity) | **YES** | Mechanical dispersion is identical for heat and solute |
| $n_e$ (effective porosity) | **YES** | Same pore space carries both heat and solute |
| $\lambda$ (thermal conductivity) | **NO** | Replace with molecular diffusion coefficient $D_m^*$ |
| Thermal retardation $R_T$ | **NO** | Replace with sorption retardation $R_s = 1 + \rho_b K_d / n_e$ |
| Source terms | **NO** | Solute sources are chemically specific |

**Key insight:** Dispersivity is the most difficult transport parameter to measure directly. By calibrating it with heat (which is "free" — temperature data is abundant), you gain a value that transfers directly to solute transport. This is the main practical advantage of heat-as-tracer calibration.

</details>

In [ ]:
# Checkpoint — Which parameter does NOT transfer from heat to solute?
create_multiple_choice('task_t05_checkpoint_transfer')

---

## 6 — Summary

In [ ]:
# Dynamic model summary
print('=' * 60)
print('TRANSPORT CALIBRATION SUMMARY')
print('=' * 60)
print(f'  Monitoring wells:    {len(obs_well_ids)}')
print(f'  Tracer test wells:   {len(tt_results)}')
print(f'  Calibration period:  {calib_years[0]}-{calib_years[-1]} ({calib_nper} months)')
print(f'  Sweep values:        {alpha_L_values} m ({len(alpha_L_values)} runs)')
print(f'')
print(f'  --- Tracer Test Estimates ---')
print(f'  Velocity:            {tt_results["v_md"].mean():.1f} m/d')
print(f'  Effective porosity:  {n_e_calib}')
print(f'  Dispersivity:        {alpha_L_tracer:.1f} m')
print(f'')
print(f'  --- Calibration Results ---')
print(f'  Best alpha_L:        {best_alpha:.0f} m')
print(f'  Initial RMSE:        {initial_rmse:.4f} C')
print(f'  Calibrated RMSE:     {calibrated_rmse:.4f} C')
print(f'  Improvement:         {(1 - calibrated_rmse/initial_rmse)*100:.1f}%')
print(f'')
print(f'  --- Key Finding ---')
print(f'  Tracer test and temperature calibration converge on alpha_L ~ 5 m')
print('=' * 60)

## What You're Taking Forward

| For Future Work | Value/Insight |
|-----------------|---------------|
| Calibrated $\alpha_L$ | ~5 m (confirmed by two independent methods) |
| Calibrated $n_e$ | ~0.18 (from tracer test moment analysis) |
| Calibration approach | Manual sweep guided by independent measurement |
| Non-uniqueness | $\alpha_L$–$n_e$ trade-off broken by tracer test |
| Parameter transfer | $\alpha_L$ and $n_e$ transfer directly to solute transport |

**Key takeaways:**
- **Two independent lines of evidence** (tracer test + temperature calibration) converge on the same parameters
- The tracer test constrains $n_e$ independently, reducing calibration to a 1D search over $\alpha_L$
- Heat calibration provides dispersivity estimates that **transfer directly to solute transport**
- Manual parameter sweeps build intuition about sensitivity; automated methods (PEST++) are taught in the flow track

---

**Review if needed:** [4_model_implementation.ipynb](4_model_implementation.ipynb) — GWF+GWE model construction and transient forcing

---

**Navigation:** [< NB4: Model Implementation](4_model_implementation.ipynb)

---

## Optional: Advanced Topics

> **Runtime warning:** The following section runs 2 additional transient models (~4 min total).

### Non-Uniqueness: The $\alpha_L$–$n_e$ Trade-Off

Can we get a similar fit with different parameter combinations? Let's test two trade-off pairs that keep the thermal signal amplitude roughly constant (higher $\alpha_L$ with lower $n_e$, or vice versa).

In [ ]:
# Test two trade-off combinations (full-period runs)
tradeoff_combos = [
    (3, 0.22, 'Low aL, high ne'),
    (8, 0.15, 'High aL, low ne'),
]

print('Non-uniqueness test:')
print(f'{"Label":<25s} {"alpha_L":>8} {"n_e":>8} {"RMSE (C)":>10}')
print('-' * 55)

# Include the best model for comparison
print(f'{"Best calibrated":<25s} {best_alpha:>8.0f} {n_e_calib:>8.2f} {best_rmse:>10.4f}')

for alpha_t, ne_t, label in tradeoff_combos:
    ws = os.path.join(CALIB_WS, f'tradeoff_aL{alpha_t}_ne{int(ne_t*100)}')
    if os.path.exists(ws):
        shutil.rmtree(ws)
    os.makedirs(ws)

    success_t, _ = build_and_run_transport(
        ws, alpha_t, ne_t,
        nper, full_periods,
        T_limmat_monthly, T_sihl_monthly, T_recharge_monthly,
        label=label)

    if success_t:
        sim_ts_t = extract_temperature_timeseries(
            ws, obs_gdf['cell_idx'].values, last_n=calib_nper)
        rmse_t = np.sqrt(np.mean((obs_ts.flatten() - sim_ts_t.flatten())**2))
        print(f'{label:<25s} {alpha_t:>8} {ne_t:>8.2f} {rmse_t:>10.4f}')
    else:
        print(f'{label:<25s} {alpha_t:>8} {ne_t:>8.2f} {"FAILED":>10}')

print()
print('The trade-off models are worse than the best single-parameter calibration.')
print('Without the tracer test to constrain n_e, many (aL, ne) combos could fit equally well.')


In [ ]:
# Checkpoint — What breaks the alpha_L-n_e trade-off?
create_multiple_choice('task_t05_checkpoint_nonunique')

## References

\[1\] Anderson, M.P. (2005). Heat as a Ground Water Tracer. *Ground Water*, 43(6), 951–968.

\[2\] Saar, M.O. (2011). Review: Geothermal heat as a tracer of large-scale groundwater flow and as a means to determine permeability fields. *Hydrogeology Journal*, 19, 31–52.

\[3\] Langevin, C.D., et al. (2022). MODFLOW 6 Description of Input and Output. U.S. Geological Survey Techniques and Methods 6-A62.

\[4\] Kreft, A. & Zuber, A. (1978). On the physical meaning of the dispersion equation and its solutions for different initial and boundary conditions. *Chemical Engineering Science*, 33(11), 1471–1480.
